# Table: Bound slope analyses — all blocks, OL and MX

Extends the Panel C and Panel D analyses from Figure 3 (and the premature-crossing check)
to all 6 blocks for both the OL (open-loop / blocked SNR) and MX (mixed SNR) datasets.

Three analyses, each computed per participant × block:

### 1. Bound ~ DT &nbsp;&nbsp;(Figure 3 Panel C)
Linear regression of |bound| on DT (DT ≥ 2).  
Tests whether inferred bounds grow systematically with decision time.  
**Null**: median slope = 0.

### 2. Δbound ~ bound &nbsp;&nbsp;(Figure 3 Panel D)
Linear regression of Δ|bound|ₜ = |bound|ₜ − |bound|ₜ₋₁ on current |bound|ₜ.  
Tests whether large bounds tend to revert toward the mean across consecutive trials.  
Slope = −1 → pure regression to mean; slope = 0 → no systematic trial-to-trial change.  
**Null**: median slope = −1.

### 3. Premature bound crossings
Fraction of good trials where the pigeon exceeded |bound| in the choice direction
*before* the NDT crossing window (steps 0 … RT − NDT − 3).  
No hypothesis test — reported as median [IQR] across participants.

---

**Table summary per block**: median [IQR] of per-participant values,
plus Wilcoxon signed-rank p-value (vs. stated null) for analyses 1 and 2.

**Data**: Both datasets loaded with `combine_snr=False` to retain all 6 raw
block / SNR combinations. Good-trial filter: DT ≥ 1, trial ≥ 10, no wall hit,
no RT below the RT cutoff. Additional DT ≥ 2 filter for analysis 1.

In [11]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import scipy.stats

from pigeon.data import get_data_table, get_good_trial_array

In [12]:
# combine_snr=False retains each block as a separate condition rather than
# pooling across SNR levels, giving one row per block in the table.
data_table_ol = get_data_table(task_type='OL', combine_snr=False)
data_table_mx = get_data_table(task_type='MX', combine_snr=False)

BLOCKS = sorted(data_table_ol['block_index'].dropna().unique().astype(int))
print('OL blocks:', BLOCKS)
print('MX blocks:', sorted(data_table_mx['block_index'].dropna().unique().astype(int)))

  1: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_14h39.24.096.csv
  2: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_15h37.48.260.csv
  3: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h34.46.761.csv
  4: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h35.25.285.csv
  5: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h36.33.669.csv
  6: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h36.41.364.csv
  7: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h36.49.086.csv
  8: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pi

In [13]:
def _premature_crossing(row):
    """
    True if the pigeon exceeded |bound| in the choice direction at any step
    before the NDT crossing window (steps[0 : RT-2-ndt_lag]).

    The NDT crossing window is the pair of steps used to infer the bound
    (steps[RT-2-ndt_lag] and steps[RT-1-ndt_lag]).  Any step strictly before
    that window that already reached |bound| in the choice direction is
    considered a premature crossing.

    Returns False for trials too short to have pre-crossing steps, nan if
    data are missing or inconsistent.
    """
    steps = row['steps']
    bound = row['bound']
    RT, DT = row['RT'], row['DT']
    if (steps is None or len(steps) == 0 or not np.isfinite(bound) or bound == 0
            or not np.isfinite(RT) or not np.isfinite(DT)):
        return np.nan
    RT = int(RT); DT = int(DT)
    ndt_lag = RT - DT - 1          # lag from RT back to the NDT crossing step
    pre_end = RT - 2 - ndt_lag     # index of first step in the NDT window; check [0:pre_end]
    if pre_end <= 0:
        return False               # trial too short — no steps before the window
    sc = steps[np.isfinite(steps)]
    if len(sc) < RT:
        return np.nan              # step array truncated, can't reliably index
    # A step counts as a premature crossing if it is in the choice direction
    # (same sign as bound) and at least as large in magnitude as the bound.
    return bool(np.any(sc[:pre_end] * np.sign(bound) >= np.abs(bound)))


def compute_slope_stats(data_table, blocks):
    """
    For each block compute per-subject statistics:
      slopes_c  : |bound| ~ DT slopes  (Figure 3 Panel C, DT >= 2)
      slopes_d  : Δ|bound| ~ |bound| slopes  (Figure 3 Panel D)
      fracs_pc  : fraction of good trials with a premature bound crossing

    All three analyses draw from the same base trial set (good-quality trials
    with a valid bound).  A subject is included in an analysis if they have
    >= 3 valid data points for that analysis.

    Returns a list of row-dicts (one per block) with median, IQR, Wilcoxon p,
    and a single participant count n shared across all analyses.
    """
    subjects = np.sort(data_table['subject_index'].dropna().unique())
    good     = get_good_trial_array(data_table)   # DT>=1, trial>=10, no wall, no short RT
    rows = []

    for block in blocks:
        # Single base filter shared by all three analyses.
        lg_base = (
            good &
            data_table['bound'].notna() &
            (data_table['block_index'] == block)
        )

        # n = subjects with >= 3 qualifying trials in this block.
        n_subj = sum(
            1 for subj in subjects
            if (lg_base & (data_table['subject_index'] == subj)).sum() >= 3
        )

        slopes_c = []
        slopes_d = []
        fracs_pc = []

        for subj in subjects:
            ls = lg_base & (data_table['subject_index'] == subj)

            # ── Analysis 1: |bound| ~ DT (DT >= 2) ──────────────────────────────
            # Regress absolute bound magnitude on decision time.
            # Require DT >= 2 so the bound is well-defined (same criterion as Fig 3).
            sub_c = data_table[ls & (data_table['DT'] >= 2)]
            if len(sub_c) >= 3:
                dts = sub_c['DT'].to_numpy(dtype=float)
                ab  = np.abs(sub_c['bound'].to_numpy(dtype=float))
                fin = np.isfinite(dts) & np.isfinite(ab)
                if fin.sum() >= 3:
                    try:
                        s, *_ = scipy.stats.linregress(dts[fin], ab[fin])
                        slopes_c.append(s)
                    except ValueError:
                        pass

            # ── Analysis 2: Δ|bound| ~ |bound| ──────────────────────────────────
            # Sort by trial number so consecutive Δ|bound| pairs are meaningful.
            # A slope of -1 would mean |bound|_t = constant (pure regression to mean);
            # slope = 0 means the change is unrelated to the current value.
            sub_d = data_table[ls].sort_values('trial_number')
            if len(sub_d) >= 3:
                ab    = np.abs(sub_d['bound'].to_numpy(dtype=float))
                curr  = ab[1:]
                delta = curr - ab[:-1]   # Δ|bound|_t = |bound|_t − |bound|_{t-1}
                fin   = np.isfinite(curr) & np.isfinite(delta)
                if fin.sum() >= 3:
                    try:
                        s, *_ = scipy.stats.linregress(curr[fin], delta[fin])
                        slopes_d.append(s)
                    except ValueError:
                        pass

            # ── Analysis 3: Premature bound crossing fraction ────────────────────
            sub_e = data_table[ls]
            if len(sub_e) >= 3:
                pc_vals  = sub_e.apply(_premature_crossing, axis=1)
                valid_pc = pc_vals.notna()
                if valid_pc.sum() >= 3:
                    fracs_pc.append(float(pc_vals[valid_pc].mean()))

        def summarize(slopes, null=0.0):
            """Median [IQR] + one-sample Wilcoxon p against null."""
            arr = np.asarray(slopes, dtype=float)
            if int(np.isfinite(arr).sum()) < 3:
                return dict(median=np.nan, q25=np.nan, q75=np.nan, p=np.nan)
            q25, med, q75 = np.percentile(arr, [25, 50, 75])
            try:
                _, p = scipy.stats.wilcoxon(arr - null)
            except ValueError:
                p = np.nan
            return dict(median=med, q25=q25, q75=q75, p=p)

        def summarize_pc(fracs):
            """Median [IQR] only — no hypothesis test for premature crossings."""
            arr = np.asarray(fracs, dtype=float)
            if int(np.isfinite(arr).sum()) < 3:
                return dict(median=np.nan, q25=np.nan, q75=np.nan)
            q25, med, q75 = np.percentile(arr, [25, 50, 75])
            return dict(median=med, q25=q25, q75=q75)

        sc = summarize(slopes_c, null=0.0)
        sd = summarize(slopes_d, null=-1.0)
        se = summarize_pc(fracs_pc)
        rows.append(dict(
            block=block,
            n=n_subj,
            c_median=sc['median'], c_q25=sc['q25'], c_q75=sc['q75'], c_p=sc['p'],
            d_median=sd['median'], d_q25=sd['q25'], d_q75=sd['q75'], d_p=sd['p'],
            e_median=se['median'], e_q25=se['q25'], e_q75=se['q75'],
        ))

    return rows

In [14]:
rows_ol = compute_slope_stats(data_table_ol, BLOCKS)
rows_mx = compute_slope_stats(data_table_mx, BLOCKS)

In [15]:
def build_table(rows, dataset):
    def fmt_slope(med, q25, q75):
        if np.isnan(med):
            return 'NA'
        return f'{med:.4f} [{q25:.4f}, {q75:.4f}]'

    def fmt_pc(med, q25, q75):
        if np.isnan(med):
            return 'NA'
        return f'{med:.3f} [{q25:.3f}, {q75:.3f}]'

    def fmt_p(p):
        if np.isnan(p):
            return 'NA'
        return f'{p:.4f}' if p >= 0.0001 else '<0.0001'

    records = []
    for r in rows:
        records.append({
            'Dataset': dataset,
            'Block\n(n)':  f"{r['block']}\n({r['n']})",
            'Premature bound crossings\nmedian [IQR]': fmt_pc(r['e_median'], r['e_q25'], r['e_q75']),
            '|bound|~DT  median [IQR]':    fmt_slope(r['c_median'], r['c_q25'], r['c_q75']),
            '|bound|~DT  p (vs 0)':        fmt_p(r['c_p']),
            'Δbound~bound  median [IQR]':   fmt_slope(r['d_median'], r['d_q25'], r['d_q75']),
            'Δbound~bound  p (vs -1)':      fmt_p(r['d_p']),
        })
    return pd.DataFrame(records)


df_ol  = build_table(rows_ol, 'OL')
df_mx  = build_table(rows_mx, 'MX')
df_all = pd.concat([df_ol, df_mx], ignore_index=True)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)
display(df_all.style.hide())

Dataset,Block (n),Premature bound crossings median [IQR],|bound|~DT median [IQR],|bound|~DT p (vs 0),Δbound~bound median [IQR],Δbound~bound p (vs -1)
OL,1 (57),"0.182 [0.131, 0.229]","0.0007 [-0.0097, 0.0087]",0.9398,"0.9926 [0.8634, 1.1181]",<0.0001
OL,2 (59),"0.200 [0.150, 0.285]","-0.0024 [-0.0065, 0.0029]",0.0506,"0.9707 [0.8311, 1.0869]",<0.0001
OL,3 (53),"0.222 [0.160, 0.303]","-0.0035 [-0.0114, 0.0040]",0.0745,"1.0087 [0.8435, 1.0708]",<0.0001
OL,4 (53),"0.044 [0.000, 0.083]","0.0074 [-0.0144, 0.0400]",0.0765,"1.0339 [0.9019, 1.1586]",<0.0001
OL,5 (57),"0.088 [0.048, 0.150]","-0.0023 [-0.0159, 0.0139]",0.8130,"1.0039 [0.8974, 1.0802]",<0.0001
OL,6 (49),"0.075 [0.030, 0.150]","-0.0011 [-0.0250, 0.0219]",0.8875,"1.0690 [0.8826, 1.1989]",<0.0001
MX,1 (60),"0.123 [0.089, 0.161]","0.0059 [-0.0035, 0.0137]",0.0148,"1.0006 [0.9199, 1.0945]",<0.0001
MX,2 (59),"0.134 [0.087, 0.198]","0.0032 [-0.0028, 0.0086]",0.0497,"0.9748 [0.8726, 1.0703]",<0.0001
MX,3 (54),"0.128 [0.070, 0.194]","-0.0023 [-0.0120, 0.0077]",0.3117,"1.0007 [0.8802, 1.1010]",<0.0001
MX,4 (54),"0.074 [0.035, 0.109]","0.0047 [-0.0072, 0.0200]",0.0400,"1.0297 [0.9246, 1.1398]",<0.0001
